In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import requests
import json
import numpy as np
import time
import pandas as pd

from src.evaluation import *
from src.bedrock import get_bearer_token
from src.prompt import *

import warnings
warnings.filterwarnings('ignore')

In [3]:
get_bearer_token()

MAX_ERRORS = 10
OPENSEARCH_DOMAIN_FQDN = 'https://vpc-int-use1-opensearch-ml-c32nkeiaudrckcr2ep7jghefje.us-east-1.es.amazonaws.com'
HEADERS = {
    'Content-Type': 'application/json',
    'Accept-Encoding': 'gzip',
}

# INDEX_NAME = 'qna-data'
INDEX_NAME = 'search-data-titan-embed-2'

query_json_path = "data/variantQna.json"
with open( query_json_path, "r" ) as file:
    test_query_data_variant = json.load(file)

query_json_path = "data/final-questions.json"
with open( query_json_path, "r" ) as file:
    test_query_data_final = json.load(file)

Bearer token set as BEARER_TOKEN_STR global variable


In [7]:
import random

random.seed(0)

In [8]:
# sampling
final_sample = random.sample(test_query_data_final, 100)

In [9]:
models = ["sonnet35_v2", "nova_pro", "nova_lite"]
eval_results = {}

In [10]:
from tqdm.auto import tqdm

In [11]:
for model in models:
    if model == "sonnet35_v2":
        results_dfs = evaluate_queries(
            query_data=final_sample,
            opensearch_domain=OPENSEARCH_DOMAIN_FQDN,
            index_name=INDEX_NAME,
            headers=HEADERS,
            top_k=5,
            embed_model="amazon.titan-embed-text-v1-pgo",
            rerank_model=model,
            rerank_prompt=rerank_prompt_claude)
    else:
        results_dfs = evaluate_queries(
            query_data=final_sample,
            opensearch_domain=OPENSEARCH_DOMAIN_FQDN,
            index_name=INDEX_NAME,
            headers=HEADERS,
            top_k=5,
            embed_model="amazon.titan-embed-text-v1-pgo",
            rerank_model=model,
            rerank_prompt=rerank_prompt_nova)

    summary_metrics = {}
    for search_type, df in results_dfs.items():
        summary_metrics[search_type] = get_summary_metrics(df)

    combined_metrics = pd.concat(summary_metrics, axis=1)
    combined_metrics.columns = combined_metrics.columns.get_level_values(0)
    eval_results[model] = combined_metrics

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

In [12]:
eval_results["sonnet35_v2"]

,hybrid_boost_0.0001,hybrid_boost_0.0001_reranked
Accuracy,0.800000,0.850000
Avg Latency,0.854317,2.814494
P10 Latency,0.436453,2.070010
P90 Latency,0.795011,3.318983
P99 Latency,4.546858,6.578082
Max Latency,13.210307,14.975327


In [13]:
eval_results["nova_pro"]

,hybrid_boost_0.0001,hybrid_boost_0.0001_reranked
Accuracy,0.800000,0.850000
Avg Latency,0.637757,1.193128
P10 Latency,0.377867,0.896654
P90 Latency,0.785712,1.416399
P99 Latency,1.828326,2.303182
Max Latency,6.261557,6.649872


In [14]:
eval_results["nova_lite"]

,hybrid_boost_0.0001,hybrid_boost_0.0001_reranked
Accuracy,0.800000,0.840000
Avg Latency,0.663051,1.249074
P10 Latency,0.419002,1.027506
P90 Latency,0.772807,1.442125
P99 Latency,1.828247,2.522554
Max Latency,2.892210,3.581965


In [15]:
for k, v in eval_results.items():
    eval_results[k] = v.add_suffix(f"_{k}")
merged_df = pd.concat(list(eval_results.values()), axis=1)
merged_df

,hybrid_boost_0.0001_sonnet35_v2,hybrid_boost_0.0001_reranked_sonnet35_v2,hybrid_boost_0.0001_nova_pro,hybrid_boost_0.0001_reranked_nova_pro,hybrid_boost_0.0001_nova_lite,hybrid_boost_0.0001_reranked_nova_lite
Accuracy,0.800000,0.850000,0.800000,0.850000,0.800000,0.840000
Avg Latency,0.854317,2.814494,0.637757,1.193128,0.663051,1.249074
P10 Latency,0.436453,2.070010,0.377867,0.896654,0.419002,1.027506
P90 Latency,0.795011,3.318983,0.785712,1.416399,0.772807,1.442125
P99 Latency,4.546858,6.578082,1.828326,2.303182,1.828247,2.522554
Max Latency,13.210307,14.975327,6.261557,6.649872,2.892210,3.581965


In [17]:
rerank_prompt_claude_prev = """
<task>
Identify the most relevant search result that best matches the intent of the query.
</task>

<instructions>
1. Exact Match Analysis (Highest Priority):
- Check if the query appears word-for-word in the result's example questions
- Match between query keywords and the result's primary topic description
- Look for exact matches in the intent description

2. Topic Focus:
- Whether the result is dedicated to answering this specific type of question
- Whether the query topic is the main focus vs. being a secondary topic
- How directly the result addresses the query subject

3. Example Questions Alignment:
- How closely the example questions match the query pattern
- Whether the example questions cover the same information type
- Whether the examples suggest the result can provide the specific information needed

4.You must return the index of the most relevant search result from 0 to {max_idx}.
</instructions>

<query>
{query}
</query>

<search_results>
{formatted_results}
</search_results>

<output_format>
Return only the index (from 0 to {max_idx}) of the best result.
Example: 2
</output_format>

Think step-by-step before returning ONLY the index without any explanation:
"""

In [18]:
rerank_prompt_nova_prev = """
##task##
Identify the most relevant search result that best matches the intent of the query.

##instructions##
1. Exact Match Analysis (Highest Priority):
- Check if the query appears word-for-word in the result's example questions
- Match between query keywords and the result's primary topic description
- Look for exact matches in the intent description

2. Topic Focus:
- Whether the result is dedicated to answering this specific type of question
- Whether the query topic is the main focus vs. being a secondary topic
- How directly the result addresses the query subject

3. Example Questions Alignment:
- How closely the example questions match the query pattern
- Whether the example questions cover the same information type
- Whether the examples suggest the result can provide the specific information needed

4.You must return the index of the most relevant search result from 0 to {max_idx}.

##query##
{query}

##search_results##
{formatted_results}

##output_format##
Return ONLY the index (from 0 to {max_idx}) of the best result.
Example: 2

Think step-by-step before returning ONLY the index without any new lines, explanation, or additional text:
"""

In [19]:
for model in models:
    if model == "sonnet35_v2":
        results_dfs = evaluate_queries(
            query_data=final_sample,
            opensearch_domain=OPENSEARCH_DOMAIN_FQDN,
            index_name=INDEX_NAME,
            headers=HEADERS,
            top_k=5,
            embed_model="amazon.titan-embed-text-v1-pgo",
            rerank_model=model,
            rerank_prompt=rerank_prompt_claude_prev)
    else:
        results_dfs = evaluate_queries(
            query_data=final_sample,
            opensearch_domain=OPENSEARCH_DOMAIN_FQDN,
            index_name=INDEX_NAME,
            headers=HEADERS,
            top_k=5,
            embed_model="amazon.titan-embed-text-v1-pgo",
            rerank_model=model,
            rerank_prompt=rerank_prompt_nova_prev)

    summary_metrics = {}
    for search_type, df in results_dfs.items():
        summary_metrics[search_type] = get_summary_metrics(df)

    combined_metrics = pd.concat(summary_metrics, axis=1)
    combined_metrics.columns = combined_metrics.columns.get_level_values(0)
    eval_results[model] = combined_metrics

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

In [20]:
for k, v in eval_results.items():
    eval_results[k] = v.add_suffix(f"_{k}")
merged_df = pd.concat(list(eval_results.values()), axis=1)
merged_df

,hybrid_boost_0.0001_sonnet35_v2,hybrid_boost_0.0001_reranked_sonnet35_v2,hybrid_boost_0.0001_nova_pro,hybrid_boost_0.0001_reranked_nova_pro,hybrid_boost_0.0001_nova_lite,hybrid_boost_0.0001_reranked_nova_lite
Accuracy,0.800000,0.840000,0.800000,0.830000,0.800000,0.850000
Avg Latency,0.641641,2.521557,0.617812,1.200290,0.657803,1.207740
P10 Latency,0.369080,1.850723,0.404548,0.937329,0.378707,0.890625
P90 Latency,0.740399,3.214538,0.789963,1.455583,0.763476,1.377741
P99 Latency,2.478326,4.817575,1.641154,2.221150,1.849404,2.467524
Max Latency,2.982943,5.053689,1.993992,2.677171,6.615199,7.051358
